### Overfitting

In this notebook we demonstrate overfitting and show some techniques to combat it. We will train a CNN classifier on the [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) dataset and improve it step by step, changing only one thing at a time:
1. A vanilla network, which overfits massively.
2. The same network with **dropout**.
3. Dropout + **batch normalization**.
4. Dropout + batch normalization + **data augmentation** (horizontal flipping). In this last run we also save the best model to disk.

Let's start with the usual imports.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'    # has to be set before keras is imported

from time import time
import numpy as np
import keras
from keras import Model
from keras.layers import Activation, BatchNormalization, Conv2D, Dense, Dropout, Flatten, Input, MaxPooling2D
from sklearn.model_selection import train_test_split

from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]

# Set the seed for reproducibility (python, numpy and the backend)
keras.utils.set_random_seed(1234)

#### Dataset

CIFAR-10 consists of 60,000 small (32x32) color images of 10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship and truck). It is one of the Keras built-in datasets, so there is nothing to download manually: 50,000 images are meant for training and 10,000 for testing.

As seen in the lecture, we need three subsets of the data: **train**, **validation** and **test**. The training set is used to fit the weights. The validation set is used to monitor the training and to take decisions (which regularization to use, when to stop, ...). The test set is used only once, at the very end, for the final evaluation. Therefore, we hold out 5,000 of the training images as the validation set. The labels stay as integer class indices, so we will use the `sparse_categorical_crossentropy` loss.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(class_names)

# Normalization
x_train = x_train.astype('float32')/255
x_test = x_test.astype('float32')/255

# Hold out a (class-balanced) validation set
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=5000, stratify=y_train, random_state=0)

print('Train set:      ', len(y_train), 'samples')
print('Validation set: ', len(y_val), 'samples')
print('Test set:       ', len(y_test), 'samples')
print('Sample dims:    ', x_train.shape)

Let's visualise some random samples.

In [ ]:
cnt = 1
for r in range(3):
    for c in range(6):
        idx = np.random.randint(len(x_train))
        plt.subplot(3, 6, cnt)
        plt.imshow(x_train[idx, ...])
        plt.title(class_names[y_train[idx]])
        cnt = cnt + 1

### 1. Vanilla Network

We start with a plain CNN classifier: two blocks of two convolutions followed by max pooling, a dense layer and the softmax classification head. It has about 1.1M parameters, which is a lot for 45,000 small images.

In [ ]:
keras.utils.set_random_seed(1234)

inputs = Input(shape=(32, 32, 3))

net = Conv2D(32, kernel_size=(3, 3), activation="relu", padding='same')(inputs)
net = Conv2D(32, kernel_size=(3, 3), activation="relu", padding='same')(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Conv2D(64, kernel_size=(3, 3), activation="relu", padding='same')(net)
net = Conv2D(64, kernel_size=(3, 3), activation="relu", padding='same')(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Flatten()(net)
net = Dense(256, activation="relu")(net)
outputs = Dense(num_classes, activation="softmax")(net)

model = Model(inputs, outputs)
model.summary()

And let's train it. Note that we pass the validation data to `fit`, so that the validation loss and accuracy are computed after every epoch.

In [ ]:
epochs = 40
batch_size = 64

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

start = time()
history_vanilla = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(x_val, y_val), verbose=2)
print('Elapsed time', time() - start)

Let's plot the history. The way to detect overfitting is to compare the training and validation curves.

In [ ]:
def plot_history(history):
    h = history.history
    epochs = range(len(h['loss']))

    plt.subplot(121), plt.plot(epochs, h['loss'], '.-', epochs, h['val_loss'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('loss')
    plt.legend(['Train', 'Validation'])
    plt.subplot(122), plt.plot(epochs, h['accuracy'], '.-',
                               epochs, h['val_accuracy'], '.-')
    plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Accuracy')
    plt.legend(['Train', 'Validation'])

    print('Train Acc     ', h['accuracy'][-1])
    print('Validation Acc', h['val_accuracy'][-1])

plot_history(history_vanilla)

The vanilla network overfits massively. The training accuracy goes to almost 99%, while the validation accuracy stalls at about 73% (after reaching its best value of about 75% in the first epochs). The validation loss is the clearest sign: it reaches its minimum of 0.75 after only a few epochs and then keeps growing to more than 2, i.e. the network becomes more and more confident in its wrong predictions. The model memorizes the training set instead of learning something general.

### 2. Dropout

**Dropout** randomly switches off a fraction $p$ of the neurons of a layer in every training step (their outputs are set to zero), and the remaining outputs are scaled by $1/(1-p)$ to keep the expected activation unchanged. A neuron cannot rely on the presence of specific other neurons, so the neurons cannot co-adapt and the network has to learn more robust features. It can also be seen as training a large ensemble of "thinned" networks that share their weights.

Dropout is active only during training. At inference time all the neurons are used (Keras takes care of switching this behaviour automatically). The layer has no trainable parameters.

We add `Dropout(0.5)` after the dense layer. This is where most of the parameters of our network are (about 1.05M of the 1.12M), and thus where most of the memorization happens. Everything else stays the same.

In [ ]:
keras.utils.set_random_seed(1234)

inputs = Input(shape=(32, 32, 3))

net = Conv2D(32, kernel_size=(3, 3), activation="relu", padding='same')(inputs)
net = Conv2D(32, kernel_size=(3, 3), activation="relu", padding='same')(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Conv2D(64, kernel_size=(3, 3), activation="relu", padding='same')(net)
net = Conv2D(64, kernel_size=(3, 3), activation="relu", padding='same')(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Flatten()(net)
net = Dense(256, activation="relu")(net)
net = Dropout(0.5)(net)                      # <- new
outputs = Dense(num_classes, activation="softmax")(net)

model = Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

start = time()
history_dropout = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(x_val, y_val), verbose=2)
print('Elapsed time', time() - start)

In [ ]:
plot_history(history_dropout)

Dropout helps. The training accuracy dropped from 99% to 95%, the validation accuracy rose to about 76% (from 73%) and the validation loss grows much slower (1.3 at the end vs. 2.2). The network still overfits, though, since the validation loss starts growing after the first 10 epochs or so.

### 3. Batch Normalization

**Batch normalization** (BN) normalizes the activations of a layer. For every channel, it subtracts the mean and divides by the standard deviation, both computed over the current mini-batch, and then applies a learnable scale $\gamma$ and shift $\beta$. This keeps the inputs of each layer in a stable range, which typically makes the training faster and more stable. The noise coming from the batch statistics also has a mild regularization effect.

BN is usually placed between the convolution and the activation: **Conv → BN → ReLU**. That is why the convolutions below have no activation anymore and the ReLU is a separate layer.

A BN layer has two kinds of parameters:
* **Trainable**: $\gamma$ and $\beta$, learned by gradient descent like any other weight.
* **Non-trainable**: the *moving mean* and the *moving variance*. They are running averages of the batch statistics, updated during the training, but not by gradient descent. Check the summary below: they are counted in the `Non-trainable params`.

At inference time we cannot use the statistics of a mini-batch (we may even process a single image), so the stored moving averages are used instead. The same train/inference duality holds for dropout. Keras switches the mode automatically in `fit`, `predict` and `evaluate`. In other frameworks, such as PyTorch, we have to do it ourselves: `model.train()` before training and `model.eval()` before inference. If we forget `model.eval()`, BN keeps using the statistics of the current batch (so the prediction for an image depends on the other images in the batch) and dropout stays active, which results in noisy and worse predictions.

In [ ]:
keras.utils.set_random_seed(1234)

inputs = Input(shape=(32, 32, 3))

net = Conv2D(32, kernel_size=(3, 3), padding='same')(inputs)
net = BatchNormalization()(net)                                  # <- new
net = Activation("relu")(net)
net = Conv2D(32, kernel_size=(3, 3), padding='same')(net)
net = BatchNormalization()(net)                                  # <- new
net = Activation("relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Conv2D(64, kernel_size=(3, 3), padding='same')(net)
net = BatchNormalization()(net)                                  # <- new
net = Activation("relu")(net)
net = Conv2D(64, kernel_size=(3, 3), padding='same')(net)
net = BatchNormalization()(net)                                  # <- new
net = Activation("relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Flatten()(net)
net = Dense(256, activation="relu")(net)
net = Dropout(0.5)(net)
outputs = Dense(num_classes, activation="softmax")(net)

model = Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

start = time()
history_bn = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(x_val, y_val), verbose=2)
print('Elapsed time', time() - start)

In [ ]:
plot_history(history_bn)

Batch normalization improves the result again, by about 2 points in validation accuracy (about 78% at the end). The training accuracy is even lower (90%) and the validation loss reaches its minimum much later (around epoch 22) and it stays at about 1 or below until the end. Note that the curves are noisier now: the batch statistics change from batch to batch and the validation accuracy jumps up and down by a few points between epochs.

The summary above confirms the parameter count: batch norm added 768 parameters to the network, out of which 384 are non-trainable (the moving means and variances of the 192 feature maps).

### 4. Data Augmentation

The most direct way of fighting overfitting is to have more training data. **Data augmentation** creates new training samples by applying transformations that do not change the label. Here we use the simplest one, **mirroring** (horizontal flip): a mirrored dog is still a dog. Note that an augmentation has to make sense for the data: mirroring would be wrong, e.g., for digits or traffic signs. There is a large variety of other augmentations (gamma correction, noise, blurring, rotation, cropping, ...).

Augmentation is applied only to the training data. We build a data generator that, in every epoch, shuffles the training set and mirrors a random half of each batch. Thus, the network practically never sees exactly the same image twice.

In [ ]:
def datagen(x, y, batch_size):
    num_samples = len(y)
    while True:
        indices = np.random.permutation(num_samples)
        for start in range(0, num_samples - batch_size + 1, batch_size):
            idx = indices[start:start + batch_size]
            x_batch, y_batch = x[idx], y[idx]            # indexing with an array makes a copy, x is not modified

            # Augmentation: mirror a random half of the batch
            flip = np.random.rand(batch_size) > 0.5
            x_batch[flip] = x_batch[flip, :, ::-1]

            yield x_batch, y_batch

The network is the same as in the previous step.

In [ ]:
keras.utils.set_random_seed(1234)

inputs = Input(shape=(32, 32, 3))

net = Conv2D(32, kernel_size=(3, 3), padding='same')(inputs)
net = BatchNormalization()(net)
net = Activation("relu")(net)
net = Conv2D(32, kernel_size=(3, 3), padding='same')(net)
net = BatchNormalization()(net)
net = Activation("relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Conv2D(64, kernel_size=(3, 3), padding='same')(net)
net = BatchNormalization()(net)
net = Activation("relu")(net)
net = Conv2D(64, kernel_size=(3, 3), padding='same')(net)
net = BatchNormalization()(net)
net = Activation("relu")(net)
net = MaxPooling2D(pool_size=(2, 2))(net)
net = Flatten()(net)
net = Dense(256, activation="relu")(net)
net = Dropout(0.5)(net)
outputs = Dense(num_classes, activation="softmax")(net)

model = Model(inputs, outputs)
model.summary()

This time we also save the best model during the training. The `ModelCheckpoint` callback evaluates the validation loss after every epoch and overwrites the file only when it improves. This way we keep the model of the best epoch, even if the network starts overfitting later on. It also works as a simple (manual) early stopping: we let the training run for all the epochs, but keep the best model. Alternatively, the `EarlyStopping` callback can stop the training itself when the validation loss stops improving.

The training data comes from the generator now, so we need to tell `fit` how many batches make an epoch.

In [ ]:
steps_per_epoch = len(y_train) // batch_size
generator = datagen(x_train, y_train, batch_size)
checkpoint = keras.callbacks.ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

start = time()
history_flip = model.fit(generator, steps_per_epoch=steps_per_epoch, epochs=epochs, validation_data=(x_val, y_val),
                         callbacks=[checkpoint], shuffle=False, verbose=2)
print('Elapsed time', time() - start)

In [ ]:
plot_history(history_flip)

Flipping the images brings the biggest improvement so far. The validation accuracy at the end is above 81% and the training accuracy is only 88%, so the gap between them is the smallest of all four runs. The validation loss is the lowest and it stays flat until the end of the training.

The curves are quite noisy, so the model from the last epoch is not necessarily the best one. This is the reason for the checkpoint: the model with the lowest validation loss was saved to `best_model.keras` during the training.

#### Evaluation

Only now, at the very end, we use the test set. The best model was saved to disk, so we can load it back with `keras.models.load_model` (the file contains the architecture and the weights) and evaluate it. For comparison, we evaluate also the model from the last epoch.

In [ ]:
best_model = keras.models.load_model('best_model.keras')

for name, m in [('Last epoch', model), ('Best model', best_model)]:
    loss, acc = m.evaluate(x_test, y_test, verbose=0)
    print(name, '| test loss %.3f | test accuracy %.3f' % (loss, acc))

Let's also look at some predictions of the best model on random test images. The title shows the ground truth and the prediction (green = correct, red = wrong).

In [ ]:
predictions = best_model.predict(x_test[:1000], verbose=0).argmax(axis=1)

cnt = 1
for r in range(3):
    for c in range(6):
        idx = np.random.randint(1000)
        plt.subplot(3, 6, cnt)
        plt.imshow(x_test[idx, ...])
        plt.title('%s / %s' % (class_names[y_test[idx]], class_names[predictions[idx]]),
                  color='green' if predictions[idx] == y_test[idx] else 'red')
        plt.axis('off')
        cnt = cnt + 1

On the test set the two models perform similarly: the accuracy is about 81% for both. The best model has a clearly lower loss (0.58 vs. 0.65), because it was saved at the point where the validation loss was the lowest. This is the benefit of saving the best model: we do not have to guess the right number of epochs in advance and we are protected against ending the training at an unlucky epoch. The test accuracy of 81% is also consistent with the validation accuracy, as we did not use the test set for any decision.

### Summary

Let's finally compare the four runs. We plot the validation curves of all of them in the same figure.

In [ ]:
histories = {'1. Vanilla': history_vanilla, '2. + Dropout': history_dropout,
             '3. + Batch norm': history_bn, '4. + Flip': history_flip}

for name, history in histories.items():
    plt.subplot(121), plt.plot(history.history['val_loss'], label=name)
    plt.subplot(122), plt.plot(history.history['val_accuracy'], label=name)

plt.subplot(121), plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Validation loss'), plt.legend()
plt.subplot(122), plt.grid(True), plt.xlabel('epochs'), plt.ylabel('Validation accuracy'), plt.legend()

for name, history in histories.items():
    h = history.history
    print('%-16s train acc %.3f | best val acc %.3f | min val loss %.3f (epoch %d)' %
          (name, h['accuracy'][-1], max(h['val_accuracy']), min(h['val_loss']), np.argmin(h['val_loss'])))

All four runs use exactly the same network and the same data, the only differences are the regularization techniques (the table above counts epochs from 0). Each of them helped:

| run | training acc. | validation acc. (last epoch) | validation loss (last epoch) |
|---|---|---|---|
| 1. Vanilla | 0.99 | 0.73 | 2.2 |
| 2. + Dropout | 0.95 | 0.76 | 1.3 |
| 3. + Batch norm | 0.90 | 0.78 | 1.0 |
| 4. + Flip | 0.88 | 0.81 | 0.6 |

The gap between the training and validation accuracy shrinks with every technique, which is exactly what we mean by fighting overfitting. Note that the differences between the runs of a single training are partially affected by randomness (initialization, batch order), especially for the noisy batch norm run, so do not read too much into a difference of one point.

There are other ways to fight overfitting that we did not try here: weight regularization (penalizing large weights), reducing the model complexity (fewer layers or neurons) and stopping the training early with the `EarlyStopping` callback. Dropout and data augmentation are, however, among the cheapest and most effective ones.